In [ ]:
import anndata as ad
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
import warnings
import os  
import loompy
import gzip
import shutil
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import mannwhitneyu

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sc.settings.verbosity = 2
sc.settings.autoshow = False
sc.settings.set_figure_params(dpi=50, dpi_save=300, format='png', 
                             frameon=False, transparent=True, fontsize=10, figsize=(4, 4))

warnings.simplefilter(action='ignore', category=FutureWarning)

plt.rcParams["image.aspect"] = "equal"
plt.rcParams["figure.figsize"] = ([4, 4])  
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.grid'] = False

colorrs = ["#4DBBD5", "#00A087", "#E64B35","#3C5488", "#F39B7F", "#8491B4",
        "#91D1C2",  "#B9C984",  "#9ACBDE", "#F494BE", "#EDCAE0", 
        "#C8CADF", "#F47892", "#F6A395",  "#C9AFA2", "#ABADC5", "#AEB9AC", 
        "#4b6aa8", "#3ca0cf", "#c376a7", "#ad98c3", "#cea5c7",
        "#53738c", "#a5a9b0", "#a78982", "#696a6c", "#92699e",
        "#d69971", "#df5734", "#6c408e", "#ac6894", "#d4c2db",
        "#537eb7", "#83ab8e", "#ece399", "#405993", "#cc7f73",
        "#b95055", "#d5bb72", "#bc9a7f", "#e0cfda", "#d8a0c0",
        "#d69a55", "#64a776", "#cbdaa9",
        "#efd2c9", "#da6f6d", "#ebb1a4", "#a44e89", "#a9c2cb",
        "#b85292", "#6d6fa0", "#8d689d", "#c8c7e1", "#d25774",
        "#c49abc", "#927c9a", "#3674a2", "#9f8d89", "#72567a",
        "#63a3b8", "#c4daec", "#61bada", "#b7deea", "#e29eaf",
        "#4490c4", "#e6e2a3",  "#c4612f", "#9a70a8",
        "#76a2be", "#408444", "#c6adb0", "#9d3b62", "#2d3462"]

In [ ]:
highlight_celltypes = ["B_01_Naive_TCL1A_IGHD","B_03_Memory_CD27","B_02_iMemory_IGHD_CD27","B_04_Plasma_IGHA1_IGHG1"]
highlight_colors = ["#4DBBD5","#00A087","#E64B35","#3C5488"]
highlight_palette = dict(zip(highlight_celltypes, highlight_colors))
palette = {ct:highlight_palette.get(ct,"lightgray") for ct in adata.obs["celltype"].astype("category").cat.categories}

fig, ax = plt.subplots(figsize=(4.3,2.5), dpi=300)
sc.pl.umap(adata, color="celltype", size=0.5, palette=palette, ax=ax, show=False, frameon=True, legend_loc=None)

handles = [Line2D([0],[0],marker="o",color="w",markerfacecolor=highlight_palette[ct],markersize=6,markeredgecolor="none") for ct in highlight_celltypes]
ax.legend(handles, highlight_celltypes, bbox_to_anchor=(1.05,0.5), loc="center left", frameon=False, fontsize=8)

ax.spines[["top","right"]].set_visible(False)
ax.set_xlabel("UMAP1", fontsize=8)
ax.set_ylabel("UMAP2", fontsize=8)
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)

fig.tight_layout()
fig.savefig("Fig.5/B_cell_subsets_UMAP.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
raw_adata = adata.raw.to_adata()
Bcell = raw_adata[raw_adata.obs['celltype_major2']=='B cells'].copy()
sc.pl.umap(Bcell)

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
from matplotlib.colors import LinearSegmentedColormap

markers = ["CD79A","CD79B","TCL1A","IGHD","CD27","IGHA1","IGHG1"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", ["#4DBBD5","white","#E64B35"], N=30)

with plt.rc_context({"axes.linewidth":0.5,"xtick.major.width":0.5,"ytick.major.width":0.5}):
    fig = sc.pl.stacked_violin(Bcell, markers, groupby="celltype", cmap=cmap, figsize=(4,2),
                               linewidth=0.5, colorbar_title="Expression", return_fig=True)

fig.savefig("Fig.S13/Bcell_stacked_violin_markers.pdf", dpi=300, bbox_inches="tight")
plt.show()

<h1>plasma</h1>

In [ ]:
plasma_cells = Bcell[Bcell.obs['celltype'] == 'B_04_Plasma_IGHA1_IGHG1']

In [ ]:
igh_genes = ["IGHG1","IGHG2","IGHG3","IGHG4","IGHA1","IGHA2","IGHM"]
colors = ["#4DBBD5","#00A087","#E64B35","#3C5488","#F39B7F","#8491B4","#91D1C2"]

genes = [g for g in igh_genes if g in plasma_cells.var_names]
expr = plasma_cells[:, genes].X
if hasattr(expr, "toarray"): expr = expr.toarray()

max_expr = expr.max(axis=1)
dominant = np.array(genes)[expr.argmax(axis=1)]
ig_summary = pd.Series(dominant[max_expr>0]).value_counts().reindex(genes, fill_value=0)
ig_summary = ig_summary[ig_summary>0] / ig_summary.sum() * 100

fig, ax = plt.subplots(figsize=(4,4), dpi=100)
radius = 0.9
wedges, _ = ax.pie(ig_summary, colors=colors[:len(ig_summary)], startangle=90,
                   wedgeprops={"edgecolor":"white","linewidth":0.5}, radius=radius)

kw = {"arrowprops":{"arrowstyle":"-","color":"black","lw":0.5},
      "bbox":{"boxstyle":"round,pad=0.3","fc":"white","ec":"black","lw":0.3}, "va":"center"}

for wedge, gene, pct in zip(wedges, ig_summary.index, ig_summary):
    ang = (wedge.theta1+wedge.theta2)/2
    x,y = radius*np.cos(np.deg2rad(ang)), radius*np.sin(np.deg2rad(ang))
    r = 1.1 if pct>5 else 1.2
    kw["arrowprops"]["connectionstyle"] = f"angle,angleA=0,angleB={ang}"
    ax.annotate(f"{gene}\n{pct:.1f}%", xy=(x,y), xytext=(r*np.sign(x),r*np.sin(np.deg2rad(ang))),
                ha="left" if x>0 else "right", fontsize=10, **kw)

ax.set_title("Ig classes of plasma cells", fontsize=12, pad=20)
ax.axis("equal")
plt.tight_layout()
plt.savefig("Fig.5/Plasma_Cells_Dominant_IGH.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
markers = ["IGHG1","IGHG2","IGHG3","IGHG4","IGHA1","IGHA2","IGHM"]
cmap = LinearSegmentedColormap.from_list("cytotoxic_cmap", ["#4DBBD5","white","#E64B35"])

dp = sc.pl.DotPlot(plasma_cells, var_names=markers, groupby="Condition", standard_scale="var",
                   categories_order=["HC","MKPP","SKPP"], figsize=(3,3))
dp.swap_axes().style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.6, grid=True,
                     smallest_dot=20, dot_max=0.4).legend(size_title="Pct. exp.", colorbar_title="Avg. exp.")

axes = dp.show(return_axes=True); ax = axes["mainplot_ax"]

for key,title in [("size_legend_ax","Pct. exp."),("color_legend_ax","Avg. exp.")]:
    if key in axes:
        axes[key].tick_params(labelsize=12)
        axes[key].set_title(title, fontsize=12)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

ax.tick_params(axis="x", labelsize=10, rotation=45)
ax.tick_params(axis="y", labelsize=10)
ax.set(xlabel="", ylabel="")

plt.tight_layout()
plt.savefig("Fig.5/IGH_expression.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
genes = ["PRDM1","XBP1","IRF4"]
condition_order = ["HC","MKPP","SKPP"]
colors = ["#4DBBD5","#00A087","#E64B35"]
pairs = [("HC","MKPP"),("MKPP","SKPP"),("HC","SKPP")]
x = np.arange(3)

expr = plasma_cells.raw[:,genes].X if plasma_cells.raw is not None else plasma_cells[:,genes].X
if hasattr(expr,"toarray"): expr = expr.toarray()

cell_df = pd.DataFrame(expr,index=plasma_cells.obs_names,columns=genes)
cell_df[["Sample","Condition"]] = plasma_cells.obs[["Sample","Condition"]]
sample_df = cell_df.groupby(["Sample","Condition"],observed=True)[genes].mean().reset_index()
plot_df = sample_df.melt(id_vars=["Sample","Condition"],value_vars=genes,var_name="Gene",value_name="Expression")

stars_dict = {}
for gene in genes:
    sub = plot_df[plot_df["Gene"]==gene]
    groups = [sub.loc[sub["Condition"]==c,"Expression"] for c in condition_order]
    kw_p = kruskal(*groups).pvalue
    dunn = sp.posthoc_dunn(sub,val_col="Expression",group_col="Condition",p_adjust="bonferroni")
    pvals = [dunn.loc[a,b] for a,b in pairs]
    stars_dict[gene] = ["***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns" for p in pvals]

fig,axes = plt.subplots(1,len(genes),figsize=(len(genes)*2.5,4),sharey=False)

for ax,gene in zip(axes,genes):
    sub = plot_df[plot_df["Gene"]==gene]
    ymin,ymax = sub["Expression"].agg(["min","max"]); yrange = ymax-ymin if ymax!=ymin else 1
    heights = [ymax+yrange*i for i in [0.15,0.30,0.45]]

    for i,(condition,color) in enumerate(zip(condition_order,colors)):
        data = sub.loc[sub["Condition"]==condition,"Expression"].to_numpy()
        ax.scatter(np.random.normal(x[i]-0.25,0.04,len(data)),data,s=28,color=color,alpha=0.8,edgecolor="white",linewidth=0.5,zorder=3)
        box = ax.boxplot(data,positions=[x[i]-0.05],widths=0.1,patch_artist=True,showfliers=False,zorder=4)
        plt.setp(box["boxes"],facecolor="white",edgecolor=color,linewidth=1.5)
        plt.setp(box["whiskers"]+box["caps"]+box["medians"],color=color,linewidth=1.5)
        if len(data)>=2:
            violin = ax.violinplot(data,positions=[x[i]+0.1],showextrema=False)
            for body in violin["bodies"]:
                body.set(facecolor=color,alpha=0.65,edgecolor="none")
                v = body.get_paths()[0].vertices; v[:,0] = np.clip(v[:,0],x[i]+0.1,np.inf)

    for (a,b),star,h in zip(pairs,stars_dict[gene],heights):
        x1,x2 = condition_order.index(a),condition_order.index(b)
        ax.plot([x1,x2],[h,h],lw=1,color="black")
        ax.text((x1+x2)/2,h,star,ha="center",va="bottom",fontsize=12,fontweight="bold" if star!="ns" else "normal")

    ax.set_title(gene,fontsize=12,fontweight="bold",pad=12)
    ax.set_xticks(x,condition_order,rotation=45,ha="center",fontsize=12)
    ax.tick_params(axis="y",labelsize=12)
    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["left","bottom"]].set_linewidth(1)
    ax.set_ylim(ymin-yrange*0.1,ymax+yrange*0.6)
    ax.grid(False)

axes[0].set_ylabel("Log normalized expression",fontsize=12)
plt.tight_layout()
plt.savefig("Fig.5/Plasma_core_TF_expression_Native_Raincloud.pdf",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
plasma_type = "B_04_Plasma_IGHA1_IGHG1"
tfh_type = "CD4T_03_Tfh_ICOS_SLAMF1"

obs = adata.obs[adata.obs["Condition"].isin(["MKPP","SKPP"])].copy()
counts = obs.groupby(["Condition","Sample"], observed=True).agg(
    Plasma_Cells=("celltype", lambda x: (x==plasma_type).sum()),
    Tfh_Cells=("celltype", lambda x: (x==tfh_type).sum()),
    Total_Cells=("celltype","size")
).reset_index()
counts.to_csv("Fig.5/Plasma_Tfh_patient_counts.csv", index=False)

def spearman_bootstrap(df, n_boot=5000, seed=42):
    x,y = df["Plasma_Cells"].to_numpy(float),df["Tfh_Cells"].to_numpy(float)
    rho,p = stats.spearmanr(x,y)
    rng = np.random.default_rng(seed)
    boot = [stats.spearmanr(x[idx],y[idx]).statistic for idx in (rng.integers(0,len(x),len(x)) for _ in range(n_boot))]
    boot = np.asarray(boot); boot = boot[np.isfinite(boot)]
    ci = np.percentile(boot,[2.5,97.5])
    return len(x),rho,ci[0],ci[1],p

results = []
for i,condition in enumerate(["MKPP","SKPP"]):
    sub = counts[counts["Condition"]==condition]
    n,rho,lo,hi,p = spearman_bootstrap(sub,seed=42+i)
    results.append([condition,n,rho,lo,hi,p])

results_df = pd.DataFrame(results,columns=["Condition","n","rho","CI_lower","CI_upper","P_raw"])
results_df["P_FDR"] = multipletests(results_df["P_raw"],method="fdr_bh")[1]
results_df.to_csv("Fig.5/Plasma_Tfh_Spearman_statistics.csv", index=False)

colors = {"MKPP":"#4DBBD5","SKPP":"#E64B35"}
files = {"MKPP":"Figure_5F_MKPP_Plasma_Tfh","SKPP":"Figure_S11B_SKPP_Plasma_Tfh"}
format_p = lambda p: "<0.001" if p<0.001 else f"{p:.3f}"

sns.set_theme(style="ticks")
for condition in ["MKPP","SKPP"]:
    sub = counts[counts["Condition"]==condition]
    res = results_df[results_df["Condition"]==condition].iloc[0]

    fig,ax = plt.subplots(figsize=(4.2,4))
    sns.regplot(data=sub,x="Plasma_Cells",y="Tfh_Cells",ci=95,ax=ax,
                scatter_kws={"s":50,"color":colors[condition],"alpha":0.9,"edgecolors":"black"},
                line_kws={"color":"#E64B35","linewidth":2})

    text = f"$n$ = {int(res['n'])}\n$\\rho$ = {res['rho']:.3f}\n95% CI: {res['CI_lower']:.3f} to {res['CI_upper']:.3f}\n$P$ = {format_p(res['P_raw'])}; FDR = {format_p(res['P_FDR'])}"
    ax.text(0.04,0.96,text,transform=ax.transAxes,ha="left",va="top",fontsize=10)
    ax.set_title("Correlation between Plasma and Tfh",fontsize=13)
    ax.set_xlabel(f"Plasma-cell count in {condition}",fontsize=11)
    ax.set_ylabel(f"Tfh-cell count in {condition}",fontsize=11)
    ax.grid(False); sns.despine(ax=ax)

    plt.tight_layout()
    output = f"Fig.5/{files[condition]}"
    plt.savefig(output+".pdf",dpi=300,bbox_inches="tight")
    plt.savefig(output+".tiff",dpi=600,bbox_inches="tight")
    plt.show()

In [ ]:
genes = ["SEC11A","SEC23IP","SEC24A","COPA","COPB1","HSPA5","CANX","ERP44","CALR","ALG2","MGAT1","ST8SIA4",
         "B4GALT3","MAN1A1","GANAB","RPN1","UBE2L6","FBXO9","PSMB8","PSMD6","PSMB10","XBP1","IRF4","FOXO1","PRDM1"]
cmap = LinearSegmentedColormap.from_list("cytotoxic_cmap", ["#4DBBD5","white","#E64B35"])

dp = sc.pl.DotPlot(plasma_cells, var_names=genes, groupby="Condition", standard_scale="var", figsize=(10,2))
dp.style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.6, grid=True, smallest_dot=20, dot_max=0.4).legend(
    size_title="Pct. exp.", colorbar_title="Avg. exp.")

axes = dp.show(return_axes=True); ax = axes["mainplot_ax"]

for key,title in [("size_legend_ax","Pct. exp."),("color_legend_ax","Avg. exp.")]:
    if key in axes:
        axes[key].tick_params(labelsize=12)
        axes[key].set_title(title, fontsize=12)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

ax.tick_params(axis="x", labelsize=12, rotation=90)
ax.tick_params(axis="y", labelsize=12)
ax.set(xlabel="", ylabel="")

plt.tight_layout()
plt.savefig("Fig.5/plasma_TFs_expression.pdf", dpi=300, bbox_inches="tight")
plt.show()

<h1>Palantir</h1>

In [ ]:
sc.pp.log1p(Bcell)
sc.pp.highly_variable_genes(Bcell, n_top_genes=1500)

In [ ]:
palantir.utils.run_diffusion_maps(Bcell, n_components=10)
palantir.utils.determine_multiscale_space(Bcell)

In [ ]:
Bcell.obs_names[Bcell.obs["celltype"] == "B_01_Naive_TCL1A_IGHD"][0]
Bcell.obs_names[Bcell.obs["celltype"] == "B_04_Plasma_IGHA1_IGHG1"][0]

In [ ]:
terminal_states = pd.Series(
    ["B_04_Plasma_IGHA1_IGHG1"],
    index=["ACCAAACGTACGCGTC-1-M01"],
)

In [ ]:
palantir.plot.highlight_cells_on_umap(Bcell, terminal_states)
plt.show()

In [ ]:
pr_res = palantir.core.run_palantir(Bcell, Naive, num_waypoints=1000, terminal_states=terminal_states)

In [ ]:
groups = ["HC","MKPP","SKPP"]
vmin, vmax = Bcell.obs["pseudotime"].agg(["min","max"])
fig, axes = plt.subplots(1,3,figsize=(6,2))

for ax, group in zip(axes,groups):
    sub = Bcell[Bcell.obs["Condition"]==group]
    sc.pl.umap(sub, color="pseudotime", ax=ax, show=False, title=group, frameon=False,
               vmin=vmin, vmax=vmax, cmap="Spectral_r")

plt.tight_layout()
plt.savefig("Fig.S11/Bcell_pseudotime_by_condition_umap_Fixed.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df_plot = Bcell.obs[["Condition","pseudotime"]].dropna()

plt.figure(figsize=(6,4.5))
sns.kdeplot(data=df_plot, x="pseudotime", hue="Condition", hue_order=["HC","MKPP","SKPP"],
            palette=["#4DBBD5","#00A087","#E64B35"], fill=True, alpha=0.3, common_norm=False, linewidth=2)

plt.xlabel("palantir_pseudotime", fontsize=12)
plt.ylabel("Density of B cells", fontsize=12)
plt.title("Shift in B cell developmental trajectory", fontsize=14)
plt.tight_layout()
plt.savefig("Fig.5/Bcell_pseudotime_density.pdf", dpi=300)
plt.show()

<h1>Naive and memory</h1>

In [ ]:
target_clusters = [
    'B_01_Naive_TCL1A_IGHD',
    'B_02_iMemory_IGHD_CD27',
    'B_03_Memory_CD27'
]
subset_cells = Bcell[Bcell.obs['celltype'].isin(target_clusters)].copy()

In [ ]:
sc.tl.rank_genes_groups(subset_cells, "Condition", 
                        method="wilcoxon",
                        groups=['MKPP', 'SKPP'],  
                        reference='HC',          
                        corr_method='benjamini-hochberg',  
                        tie_correct=True)   
result = sc.get.rank_genes_groups_df(subset_cells, group=['MKPP', 'SKPP'])

In [ ]:
logfc_threshold, pval_threshold = 1.5, 0.01
results, up_genes = {}, {}

for group in ["MKPP","SKPP"]:
    df = sc.get.rank_genes_groups_df(subset_cells, group=group)
    sig = df[(df["pvals_adj"]<pval_threshold) & (df["logfoldchanges"].abs()>logfc_threshold)]
    results[group] = sig
    up_genes[group] = set(sig.loc[sig["logfoldchanges"]>0,"names"])

common = up_genes["MKPP"] & up_genes["SKPP"]
mkpp_unique = up_genes["MKPP"] - up_genes["SKPP"]
skpp_unique = up_genes["SKPP"] - up_genes["MKPP"]

pd.DataFrame({"gene":sorted(common)}).to_csv("Fig.S13/Bcell_common_upregulated_genes.csv", index=False)
pd.DataFrame({"gene":sorted(mkpp_unique)}).to_csv("Fig.5/Bcell_MKPP_unique_upregulated_genes.csv", index=False)
pd.DataFrame({"gene":sorted(skpp_unique)}).to_csv("Fig.5/Bcell_SKPP_unique_upregulated_genes.csv", index=False)

plt.figure(figsize=(4,3))
venn = venn2_unweighted(subsets=(len(mkpp_unique),len(skpp_unique),len(common)),
                        set_labels=("MKPP vs HC","SKPP vs HC"), set_colors=("#4DBBD5","#E64B35"), alpha=0.9)

plt.title("log2FC > 1.5, adjusted P < 0.01", fontsize=14, fontweight="bold", pad=10)
for text in list(venn.set_labels)+list(venn.subset_labels):
    if text:
        text.set_fontsize(12)
        text.set_fontweight("bold")

plt.tight_layout()
plt.savefig("Fig.5/Bcell_Venn_upregulated_genes.pdf", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()